### 과제1.

- 사용자의 취향을 기억하는 영화 추천 챗봇을 구축하세요.
- 챗봇이 갖춰야 할 기능:
  - 사용자가 좋아하는 장르를 기억
  - 이미 시청한 영화를 기억
  - 대화 기록을 기반으로 개인화된 추천 제공

#### 요구사항

- 대화 기록을 저장하기 위한 messages 리스트를 구현하세요.
- 사용자 메시지(role: "user")와 AI 응답(role: "assistant")을 모두 append하세요.
- 이미 본 영화는 추천하지 않도록 메모리를 활용해야 합니다.

#### 챗봇 테스트

```
Jupyter Notebook에서 다음과 같은 대화를 보여주세요:
User: 나는 SF 영화를 좋아해
AI: 좋은 취향이시네요! SF에는 명작이 정말 많죠...
User: 인셉션이랑 인터스텔라는 이미 봤어
AI: 좋은 선택이셨네요! 이미 보셨으니까...
User: 오늘 밤에 뭐 볼지 추천해 줄래?
AI: SF를 좋아하시고, 인셉션과 인터스텔라는 이미 보셨으니까 추천드리자면...
User: 내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?
AI: SF를 좋아하신다고 했죠! 인셉션과 인터스텔라를 보셨습니다.
```


In [19]:
import openai, json
import requests

# .env 의 OPENAI_API_KEY 를 자동으로 읽어서, openai.OpenAI() 객체 생성
client = openai.OpenAI()
messages = []

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

In [20]:
GREEN = "\033[32m"
RED = "\033[31m"
BLUE = "\033[34m"
YELLOW = "\033[33m"
PINK = "\033[35m"
RESET = "\033[0m"

In [21]:
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return response.json()


def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return response.json()


def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return response.json()


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [22]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "A function to get the popular movies",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "A function to get the details of a movie by its ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The ID of the movie",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "A function to get the cast and crew of a movie by its ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The ID of the movie",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [23]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(GREEN + f"Calling function: {function_name} with {arguments}" + RESET)

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            # print(
            #     GREEN
            #     + f"Ran {function_name} with args {arguments} for a result of {result}"
            #     + RESET
            # )

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": (
                        json.dumps(result) if not isinstance(result, str) else result
                    ),
                }
            )

        call_ai()  # 모든 tool 결과를 추가한 후에 한 번만 호출
    else:
        messages.append({"role": "assistant", "content": message.content})
        print("=" * 30)
        print(BLUE + f"AI: {message.content}" + RESET)


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [24]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    # 셀 실행희 Enter 값이 stdin 버퍼에 남아 발생하는 문제 방지
    elif not message.strip():
        continue
    else:
        messages.append({"role": "user", "content": message})
        print("=" * 30)
        print(YELLOW + f"User: {message}" + RESET)
        call_ai()

User: 나는 SF 영화를 좋아해
AI: SF 영화는 정말 흥미롭고 상상력이 풍부한 장르죠! 최근에 인기 있는 SF 영화를 알고 싶으신가요?
User: 인셉션이랑 인터스텔라는 이미 봤어
Calling function: get_popular_movies with {}
AI: 다음은 최근 인기 있는 SF 영화 몇 편입니다:

1. **Mercy**
   - **개요**: 가까운 미래, 한 형사는 아내를 살해한 혐의로 재판에 서게 됩니다. 그는 자신의 무죄를 입증하기 위해 그가 한때 지지했던 첨단 AI 판사와의 90분간의 대결을 벌입니다.
   - **개봉일**: 2026-01-20
   - **평점**: 7.1
   - ![포스터](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **28 Years Later: The Bone Temple**
   - **개요**: Dr. Kelson은 세상을 바꿀 수 있는 shocking한 새 관계에 빠지게 되며 지옥 같은 상황에 처하게 됩니다.
   - **개봉일**: 2026-01-14
   - **평점**: 7.2
   - ![포스터](https://image.tmdb.org/t/p/w780/kK1BGkG3KAvWB0WMV1DfOx9yTMZ.jpg)

3. **Greenland 2: Migration**
   - **개요**: 가족이 지구의 재앙으로부터 안전한 그린란드 벙커에 도착하지만, 새로운 고향을 찾기 위해 위험한 여정을 떠나야 합니다.
   - **개봉일**: 2026-01-07
   - **평점**: 6.5
   - ![포스터](https://image.tmdb.org/t/p/w780/z2tqCJLsw6uEJ8nJV8BsQXGa3dr.jpg)

4. **Space/Time**
   - **개요**: 치명적인 테스트 후, 과거의 영광을 되찾기 위해 범죄 세계에 발을 들여놓은 과학자들이 인류를 구하고 파괴할 수 있는 금지된 공간 변형 엔진을 

KeyboardInterrupt: Interrupted by user